# Buffalo 311 cleaning (ops 2-6)
CSE 487 Phase 1, CTRL-Freaks

our question: are 311 response times fair across buffalo council districts / neighborhoods (2020-2026)?

this notebook is my part of the cleaning, ops 2 thru 6:
- Op 2: removing irrelevant rows + columns
- Op 3: data type conversion
- Op 6: fixing swapped / fake lat and lon
- Op 4: handling missing data
- Op 5: removing duplicates

op 1 (integration) is jayda's, and op 7 (category harmonization) is someone elses. also yes, op 6 runs b4 op 4 on purpose, fixing the coords makes new missing values so op 4 has to come after to deal w them

data sources:
- 311 Service Requests (July 2008 - May 2024): https://data.buffalony.gov/Quality-of-Life/311-Service-Requests-July-2008-May-2024-/whkc-e5vr
- 311 Service Requests (May 2024 - Present): https://data.buffalony.gov/Quality-of-Life/311-Service-Requests-May-2024-Present-/3tj7-3tdz

In [ ]:
import os
import numpy as np
import pandas as pd

## Loading the data
pulling both datasets off the buffalo open data api as csv. i load everything as strings (`dtype=str`) on purpose, so pandas doesnt guess the types wrong. like case numbers such as `00971777` would loose the leading zeros. the real types get set in op 3

In [ ]:
# old one is huge (1.1M rows back to 2008) so i filter to 2020+ right in the url
# so we dont download 12 yrs we dont need. new one is only aprox 200k, so js grab all of it
oldUrl = "https://data.buffalony.gov/resource/whkc-e5vr.csv?$where=open_date%20%3E%3D%20%272020-01-01%27&$limit=2000000"
newUrl = "https://data.buffalony.gov/resource/3tj7-3tdz.csv?$limit=2000000"

os.makedirs("../data", exist_ok=True)

# only download once, after that read the saved csv bc the download takes a min
if os.path.exists("../data/raw_old.csv"):
    oldDf = pd.read_csv("../data/raw_old.csv", dtype=str)
else:
    oldDf = pd.read_csv(oldUrl, dtype=str)
    oldDf.to_csv("../data/raw_old.csv", index=False)

if os.path.exists("../data/raw_new.csv"):
    newDf = pd.read_csv("../data/raw_new.csv", dtype=str)
else:
    newDf = pd.read_csv(newUrl, dtype=str)
    newDf.to_csv("../data/raw_new.csv", index=False)

print("old:", oldDf.shape)
print("new:", newDf.shape)

## Op 2: Removing irrelevant rows and columns

**rows**
- the old dataset goes all the way back to 2008 (1,129,274 rows). we only need 2020-2026, so the api query filters to `open_date >= 2020-01-01` and that gets it down to 362,538. new dataset starts 5/11/2024, so all of it counts
- 4 rows in the old data have department (`subject`) = `"Test"`. those are test entrys, not real requests, so they go

**columns** (old 34 to 16, new 28 to 16). dropped these bc theyre repeats or useless for our question:
- `location`, `x_coordinate`, `y_coordinate`: same info as latitude/longitude, js a diff format
- 2010 census cols, `tractce20`, `geoid20_blockgroup`, `geoid20_block`, census block/block group: way to specific, block level is smaller then we need
- `city`, `state`: its all Buffalo NY lol
- `address_line_2` (91% empty), `secondarystreet` (92% empty), `division` (33% empty)
- `reason`, `object_type`: pretty much repeat `subject` and `type`
- `statusdescription`: free text notes, not usefull here
- `property_id`, `reportedlocation`, `planningsector`: parcel ids, repeat address text, and a bigger version of neighborhood

keeping census tract (`geoid20_tract` in old, `census_tract` in new) bc census population data is by tract, so later we can do requests per person instead of js raw counts. bigger districts will obviously have more requests otherwise

keeping `council_district_2011` for now bc op 4 needs it, and `duplicate_request` bc op 5 needs it

In [ ]:
print("old shape b4:", oldDf.shape)
print("new shape b4:", newDf.shape)

# double checking the api date filter actually worked
print("earliest old date:", oldDf["open_date"].min())
print("earliest new date:", newDf["createddate"].min())

In [ ]:
# test rows r not real requests
testRows = oldDf["subject"] == "Test"
print("test rows:", testRows.sum())
oldDf = oldDf[~testRows]

# only keeping what we need for the equity question: what, when, where, who handled it
oldKeep = ["case_reference", "open_date", "closed_date", "status", "subject", "type",
           "address_number", "address_line_1", "zip_code", "latitude", "longitude",
           "council_district", "council_district_2011", "police_district", "neighborhood", "geoid20_tract"]
newKeep = ["casenumber", "createddate", "closeddate", "status", "department", "type",
           "assessednumber", "assessedstreet", "zip", "latitude", "longitude",
           "council_district", "policedistrict", "planningneighborhood", "duplicate_request", "census_tract"]

oldDf = oldDf[oldKeep]
newDf = newDf[newKeep]

print("old shape after:", oldDf.shape)
print("new shape after:", newDf.shape)

### Temporary merge (stand in for op 1)
op 1 is jayda's, i js need one table to run my stuff on. so this renames both to the same col names and stacks them. **swap this cell out for jayda's merged output once shes done.** the `source` col keeps track of which system a row came from

In [ ]:
# NOTE this is jayda's op, js a quick stand in so i can test my stuff. replace w her merged file later
oldDf = oldDf.rename(columns={
    "case_reference": "caseId", "open_date": "createdDate", "closed_date": "closedDate",
    "subject": "department", "address_number": "addressNumber", "address_line_1": "street",
    "zip_code": "zip", "council_district": "councilDistrict",
    "council_district_2011": "councilDistrict2011", "police_district": "policeDistrict",
    "geoid20_tract": "censusTract"})
newDf = newDf.rename(columns={
    "casenumber": "caseId", "createddate": "createdDate", "closeddate": "closedDate",
    "assessednumber": "addressNumber", "assessedstreet": "street",
    "council_district": "councilDistrict", "policedistrict": "policeDistrict",
    "planningneighborhood": "neighborhood", "duplicate_request": "duplicateRequest",
    "census_tract": "censusTract"})

oldDf["source"] = "old"
newDf["source"] = "new"

df = pd.concat([oldDf, newDf], ignore_index=True)
print(df.shape)
df.head()

## Op 3: Data type conversion
everything is a string rn. converting:
- `createdDate`, `closedDate` to datetime (need this to get response time later)
- `latitude`, `longitude` to float
- `censusTract` to the same 6 digit code in both. new data has codes like `002400`, but the old col is the full 2020 census id like `36029002400` (36 = NY, 029 = erie county, last 6 = the tract). so for old rows i js take the last 6 digits. anything thats not 6 digits after that (UNKNOWN, or aprox 3k broken 4-5 digit codes in the new data) becomes NaN
- `duplicateRequest` to True/False. its literally the text `"true"`/`"false"`, and old rows have nothing bc the old system didnt have this flag

using `errors="coerce"` so anything that cant parse turns into NaT/NaN instead of crashing, then i count how many that happend to

In [ ]:
print("types b4:")
print(df.dtypes)

In [ ]:
df["createdDate"] = pd.to_datetime(df["createdDate"], errors="coerce")
df["closedDate"] = pd.to_datetime(df["closedDate"], errors="coerce")
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

# old rows have nothing here so they js become False
df["duplicateRequest"] = df["duplicateRequest"] == "true"

print("types after:")
print(df.dtypes)

# createdDate should have 0 failures. closedDate NaT = still open cases, thats expected not a parse fail
print("createdDate that failed to parse:", df["createdDate"].isna().sum())
print("closedDate empty (open cases):", df["closedDate"].isna().sum())

In [ ]:
# census tract is written 2 diff ways, old = full 11 digit id "36029002400", new = "002400"
# last 6 digits of the old one is the same code so js chop it
fromOld = (df["source"] == "old") & (df["censusTract"].str.len() == 11)
df.loc[fromOld, "censusTract"] = df.loc[fromOld, "censusTract"].str[-6:]

# anything not 6 digits by now is junk (UNKNOWN or those weird short codes)
badTract = df["censusTract"].notna() & (df["censusTract"].str.len() != 6)
print("bad tract codes set to NaN:", badTract.sum())
df.loc[badTract, "censusTract"] = np.nan

print("distinct tracts:", df["censusTract"].nunique())
print(df["censusTract"].value_counts().head())

## OP 8 Data Input String Cleaning
The **case_ids** have 3 distinct lengths: 17, 10, 8 where they correspond to x rows, y rows, z rows

The data in the column **department** has different inputs that represent the same department, so I decided to group them. For example, 'Parking' is the same thing as 'Dept of Parking'



In [ ]:
#17385, 345149, 202728 rows of cases_ids are of length 17,10,8
# standarizing the case_ids to all length 8
incorrect_case_ids = (df['caseId'].str.len() !=8)
print("incorrect case id length before: " + f'{incorrect_case_ids.value_counts().to_dict().get(False)}')
df.loc[incorrect_case_ids,'caseId'] = df.loc[incorrect_case_ids,'caseId'].str[-8:]
correct_case_ids = (df['caseId'].str.len() !=8)
print("incorrect case id length after: " + f'{correct_case_ids.value_counts().to_dict().get(True)}')

In [ ]:
# Getting rid of (Req_Serv) in the type column because it's reduntant
df['type'] = df['type'].str.replace('(Req_Serv)','')


In [ ]:
df['department'].unique()
# multiple data enteries represent the same thing, so they should be standarized to
#Mayor's Office -> Office of the Mayor
#Parking -> Dept of Parking
#Dept of Public Works & Public Works, Parks & Streets
#-> Department of Public Works, Parks, & Steets
# Community Services & Rec.Program & Community Services & Recreational Programming
# -> Community Services & Recreational Programming
# DPIS & Permit & Inspection Services -> Permit & Inspection Services

updated_request_map = {"Major's Office": "Office of the Mayor",
                       "Parking" : "Parking Department",
                       "Dept of Public Works": "Public Works, Parks, & Streets Department",
                       "Public Works, Parks & Streets" : "Public Works, Parks, & Streets Department",
                       "Community Services & Rec. Program.": "Community Service & Recreational Programming",
                       "Community Services & Recreational Programming": "Community Service & Recreational Programming",
                       "DPIS" : "Permit & Inspection Services",
                       "nan" : "Unknown",
                       "Dept of Law" : "Law Department"
}
df['department'] = df['department'].replace(updated_request_map)


## Op 6: Fixing swapped / fake coordinates
found 2 problems while messing w the lat/lon cols:

1. **lat and lon swapped (new data).** aprox 10k rows have latitude around -78.8 and longitude around 42.9, so they got flipped. buffalo latitude is positive (like 42.9), so any row w negative latitude is flipped. js swap them back
2. **fake placeholder coords (old data).** about 85% of old rows (basically all of 2020-2023) have *exactly* (43.0, -79.0). thats not a real spot, its a default the old system filled in. if we left it, 300k requests would all sit on one fake dot on the map, so they get set to NaN. (those rows still have neighborhood / council district so theyre still useable)

then a **bounding box check**: anything still outside roughly buffalo (lat 42.82 to 42.97, lon -78.92 to -78.79, i picked these from the range of the real coords) gets set to NaN

In [ ]:
# aprox 10k rows in the new data have lat and lon flipped, buffalo lat is +42ish so negative lat = flipped
swapped = df["latitude"] < 0
print("rows w/ lat and lon flipped:", swapped.sum())
print(df.loc[swapped, "source"].value_counts())

tempLat = df.loc[swapped, "latitude"]
df.loc[swapped, "latitude"] = df.loc[swapped, "longitude"]
df.loc[swapped, "longitude"] = tempLat

print("rows w/ negative lat after fix:", (df["latitude"] < 0).sum())

In [ ]:
# 43, -79 exactly shows up 300k+ times, thats not a real spot its a placeholder
print(df["latitude"].value_counts().head(3))

placeholder = (df["latitude"] == 43) & (df["longitude"] == -79)
print("placeholder coord rows:", placeholder.sum())
print(df.loc[placeholder, "createdDate"].dt.year.value_counts().sort_index())

df.loc[placeholder, "latitude"] = np.nan
df.loc[placeholder, "longitude"] = np.nan

In [ ]:
# rough box around buffalo, anything outside is prob bad data
inBuffalo = df["latitude"].between(42.82, 42.97) & df["longitude"].between(-78.92, -78.79)
outside = df["latitude"].notna() & ~inBuffalo
print("coords outside buffalo:", outside.sum())

df.loc[outside, "latitude"] = np.nan
df.loc[outside, "longitude"] = np.nan

print("rows w/ real coords now:", df["latitude"].notna().sum(), "/", len(df))

## Op 4: Handling missing data
both systems write the text `"UNKNOWN"` (and a couple `"UnAssigned"`) instead of leaving it blank. pandas doesnt count that as missing, so `isna()` way undercounts. so step 1 is turning those into real NaN, then deciding what to do col by col

what i decided:
- **`councilDistrict`**: the old system left this UNKNOWN for almost all of 2020-2021 (aprox 150k rows!). but `council_district_2011` is filled for most of those, so i fill the gaps from that and then drop it. catch: 2011 district lines arent exactly the same as the current ones. where both are known they match aprox 90% of the time, so its a good fill but not a perfect one, should mention in the report
- **rows still missing `councilDistrict`**: dropped. the whole point is comparing districts, so no district = cant use it
- **`closedDate` missing**: these are still open cases, not bad data. keeping them w a new `isOpen` flag so the EDA can skip them for response time stuff
- **lat/lon, zip, neighborhood, police district, address**: left as NaN. not every analysis needs them, so dropping those rows would throw away alot of good data

In [ ]:
# UNKNOWN is basically NaN in disguise, isna() cant see it
missingBefore = df.isna().sum()
unknownBefore = (df == "UNKNOWN").sum() + (df == "UnAssigned").sum()

df = df.replace(["UNKNOWN", "UnAssigned", ""], np.nan)

missingTable = pd.DataFrame({
    "isna b4": missingBefore,
    "UNKNOWN text": unknownBefore,
    "real missing now": df.isna().sum(),
    "% missing": (df.isna().sum() / len(df) * 100).round(2)})
missingTable

In [ ]:
# old system js didnt fill council district for most of 2020-2021, but the 2011 col has it
print("council district missing b4:", df["councilDistrict"].isna().sum())
df["councilDistrict"] = df["councilDistrict"].fillna(df["councilDistrict2011"])
print("council district missing after fill:", df["councilDistrict"].isna().sum())

df = df.drop(columns=["councilDistrict2011"])

In [ ]:
# no district = cant compare it to anything so it goes
noDistrict = df["councilDistrict"].isna()
print("rows w/ no district, dropping:", noDistrict.sum())
df = df[~noDistrict]

# open cases r fine, js flag them so response time stuff can skip them
df["isOpen"] = df["closedDate"].isna()
print("open cases kept:", df["isOpen"].sum())
print("rows left:", len(df))

## Op 5: Removing duplicates
checked the obvious stuff first: no exact duplicate rows, and every case id is unique (the date ranges dont overlap either, old ends 5/10/2024 and new starts 5/11/2024). so a plain `drop_duplicates()` would remove **0** rows. the real dupes are the *same issue reported more then once*:

1. **flagged dupes**: the new system marks these itself (`duplicateRequest = True`), so those go
2. **near dupes**: same request type, same address, same day. like the old data has the same "Quality of Life" complaint at 530 Rhode Island twice, 2 min apart. keeping the first one. rows w no street get skipped for this check, otherwise every no-address request of the same type on the same day would get squished into one

this matters bc dupes inflate the request counts for some areas, which would mess up comparing districts

In [ ]:
print("exact duplicate rows:", df.duplicated().sum())
print("duplicate case ids:", df["caseId"].duplicated().sum())

In [ ]:
# new system flags its own dupes
flagged = df["duplicateRequest"] == True
print("flagged as duplicate by the city:", flagged.sum())
df = df[~flagged]
df = df.drop(columns=["duplicateRequest"])

In [ ]:
# same type same address same day = almost def the same thing reported twice
# sort by time first so keep first = the earliest report
df = df.sort_values("createdDate")
df["createdDay"] = df["createdDate"].dt.date

nearDupes = df.duplicated(subset=["type", "addressNumber", "street", "createdDay"], keep="first")
# no street = cant tell if its the same spot, so dont count those
nearDupes = nearDupes & df["street"].notna()

print("near duplicates:", nearDupes.sum())
print(df.loc[nearDupes, "source"].value_counts())

df = df[~nearDupes]
df = df.drop(columns=["createdDay"])
df = df.reset_index(drop=True)
print("rows left:", len(df))

In [ ]:
df.head()
df['type'].unique()

## Save + final checks
saving the cleaned table for op 7 (category harmonization) and the EDA. then a few asserts, to make sure each op actually did what i said it did

In [ ]:
df.to_csv("../data/cleaned_311.csv", index=False)

# sanity checks, if any of these fail smth is cooked
assert (df["latitude"] < 0).sum() == 0
assert df["latitude"].dropna().between(42.82, 42.97).all()
assert (df == "UNKNOWN").sum().sum() == 0
assert df["councilDistrict"].isna().sum() == 0
assert str(df["createdDate"].dtype).startswith("datetime64")
assert str(df["closedDate"].dtype).startswith("datetime64")
assert df["caseId"].duplicated().sum() == 0
assert (df["censusTract"].dropna().str.len() == 6).all()

print("all checks passed!")
print("final shape:", df.shape)
print(df["source"].value_counts())